# Color Scales

## Introduction: Why Color Scales

**Statistical Intent:** Statistical intent in information visualization refers to the specific analytical objective a designer aims 
to achieve when mapping data to a visual attribute, such as color. It acts as a bridge between the data type (the nature of the data) 
and the intended message (the conclusion the user should draw). Choosing a color scale without considering the statistical intent is 
the most common reason for misleading visualizations.

**Note:** The Expressiveness Principle states that a visual encoding should represent all the relevant statistical features 
of the data, but not more than that. This ensures that the viewer’s mental model aligns with the underlying data distribution. 
Reference: Automating the design of graphical presentations of relational information, Jock Mackinlay, 1986. We can interpret t
he *statistical intent* as an implementation of the Expressiveness Principle. 

1. *Sequential Intent (Numerical Magnitude)*
    - Data Type: Quantitative (Ratio/Interval) or Ordinal.
    - Goal: To show the relative size or amount of a single variable.
    - Visual Logic: A smooth transition in Luminance (light to dark) or Chroma (dull to vivid).
    - Example: A map of population density or a heatmap of website clicks.
    - Design Rule: The eye should easily rank the values. Lower values are typically lighter; higher values are darker and more saturated.

2. *Diverging Intent (Deviation from a Norm)*
    - Data Type: Quantitative with a meaningful Zero or Reference point.
    - Goal: To highlight values that are "above" or "below" a specific threshold.
    - Visual Logic: Two sequential scales meeting at a neutral midpoint (usually white, light gray, or yellow).
    - Example: Budget surplus vs. deficit, temperature anomalies (warmer/colder than average), or political swing (Left vs. Right).
    - Design Rule: The midpoint must be visually quiet so that the extremes pop.

3. *Qualitative Intent (Categorical Identity)*
    - Data Type: Nominal.
    - Goal: To distinguish between different groups that have no inherent mathematical order.
    - Visual Logic: Maximum variation in Hue, while keeping Luminance and Chroma as constant as possible.
    - Example: Different types of fruit, different departments in a company, or different subway lines.
    - Design Rule: Avoid implying an order. If one color is much brighter than the others, the viewer will subconsciously think that 
    category is more important or higher.
4. *Threshold/Binary Intent (Critical Cutoff)*
    - Data Type: Quantitative mapped to a Boolean outcome.
    - Goal: To show a pass/fail state or a critical limit.
    - Visual Logic: High contrast at a specific value.
    - Example: A Warning color (Read) and Safe color (Green)

## Why this Matters: 
If one treats the d3 scale $f: D \rightarrow R$ purely as a mathematical exercise, one might introduce gradients (unintended strong
variations in color) that violates the statistical intent:

- The “Rainbow” Error occurs when using a rainbow to represent sequential data. Since the yellow part of a rainbow is significantly 
brighter than the blue part, it creates “visual artifacts” (the eye perceives a peak in the data where there is only a change in hue). 
This violates the Sequential Intent.

- The Luminance error occurs when using a qualitative scale to represent data. For instance, if Accounting is depicted as bright 
yellow and Sales as dark purple, Accounting may feel more valuable than Sales even though the data is merely labels. This violates 
the Qualitative Intent principle.

**Key Takeaway:** Statistical intent pursues that the perceptual hierarchy of your colors aligns with the mathematical hierarchy 
of your data.

**Summarizing: The Influence of Design Systems**

Modern visualization tools have categorized these "scales" based on their statistical intent, which has standardized the term:

<table>
    <tbody>
        <tr>
            <td>
                <p>
                    Scale Type<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Data Type<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Purpose<!--Break-->
                </p>
            </td>
        </tr>
        <tr>
            <td>
                <p>
                    Sequential Scale<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Quantitative / Ordinal&nbsp;<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Showing a range from low to high (e.g., light blue to dark blue).<!--Break-->
                </p>
            </td>
        </tr>
        <tr>
            <td>
                <p>
                    Diverging Scale<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Quantitative<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Highlighting a 
                    <em>neutral midpoint</em>
                     (e.g., Red-White-Blue for temperature).<!--Break-->
                </p>
            </td>
        </tr>
        <tr>
            <td>
                <p>
                    Qualitative Scale<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Nominal<!--Break-->
                </p>
            </td>
            <td>
                <p>
                    Distinguishing between categories (e.g., different colors for different fruits).<!--Break-->
                </p>
            </td>
        </tr>
    </tbody>
</table>

### Perceptual Accuracy

By talking about Color Scales, especially in HCL space, the focus shifts to Luminance and Chroma—ensuring that the scale is 
perceptually balanced.

## Transfer Function: From Data to Optical Properties

A transfer function is a mapping that assigns optical properties, such as color (R, G, B) and opacity (α), to scalar data values (v).

*Mathematically*, it is a multi-dimensional mapping:
$$
f(v) \rightarrow (r,g,b,\alpha), \quad 
\text{as vector valued function} \quad 
f(v) = \begin{pmatrix} f_r(v)\\f_g(v)\\f_b(v)\\f_\alpha(v) \end{pmatrix}
$$
Where:
$v$ is the scalar value (e.g., bone density in a CT scan or temperature in a weather model), and 
$[r, g, b]$  defines the emissive 
or reflective color and $\alpha$  (Alpha/Opacity) defines how much of the data at that value is visible or solid.
Note: The transfer function is a vector-valued function. The transfer aspect refers to assigning a visual signal to raw data. 
We are not merely mapping colors; we are defining visibility. 

### Conceptual Distinctions

- Color Palette: A static set of colors.
- Color Scale: A mathematical mapping of $[Data \rightarrow Color]$.
- Transfer Function: A sophisticated mapping of  $[Data \rightarrow Color+Opcacity]$, allowing for the exploration of 3D 
structures and internal densities.

## Color Scale

Mathematically, a d3 scale is an Injective Function that maps a value from a data space, the domain to a visual space, the range.

### Formal Definition

A scale $f$ is a function that maps an element $d$ from the Domain $D$ to an element $r$ in the Range $R$:

$$
f: D \rightarrow R
$$

- The Domain $D$: This is the Data Space. It represents the extent of your input data (e.g., temperature from  −20  to 100).
- The Range $R$: This is the Visual Space. It represents the visual attribute, e.g colors from dark blue to light blue.

### Implementation

A possible realization of a color scale consists of two steps: Normalization and Interpolation

#### Step 1: Normalization

The scale transforms the data value $d$ into a normalized value $u$, typically in the interval  $[0,1]$. For a linear scale:

$$
u(d) = \frac{d - d_{min}}{d_{max} - d_{min}}
$$

#### Step 2: Interpolation (The Visual Space)

The scale interpolate a point (color) in the visual range by interpolation with an array of colors $[c_0,\dots, c_n]$:

$$
f(d)=\
𝑆(𝑑)=\text{interp}([c_0,\dots,c_n],u(d))
$$

### Why it is called a Scale

The term scale refers to Measurement Scales (On the Theory of Scale of Measurements, S. S. Stevens, 1946):

- Continuous Scales (Linear, Log, Power): For interval or ratio data
- Discrete/Ordinal Scales (Band, Point): For ordinal data
- Categorical Scales: For nominal data. An injective map between a set of labels and a set of colors. In order to be injective 
the set of colors has to have a larger cardinality.

## Example Categorical Data

Map the set of labels ```{Spring, Summer, Autum, Winter}``` to the following range of categorical colors:

<img src="./figures/categoric_example_map.png" width="400">

A possible injective mapping can be to assign the first color to ```Spring```, the second to ```Summer```, and so on. 

## Example Continuous Color Scale

Map the domain \([ height_{min}, height_{maxt}]\) to a continuous color scale by interpolating with an array of RGB colors.

<img src="./figures/pathlines_04.png" width="400">



## Quick Reference
<table data-path-to-node="15">
<thead><tr><th>Data Type</th>
<th>Terminology for Scale</th>
<th>Visual Result</th></tr>
</thead>
<tbody>
<tr>
<td>Nominal</td>
<td>Categorical / Qualitative</td>
<td>Distinct Hues (No order).</td>
</tr>
<tr>
<td>Ordinal</td>
<td>Discrete / Sequential</td>
<td>Stepped colors (Clear order).</td>
</tr>
<tr>
<td>Quantitative (Scalar)</td>
<td>Continuous / Sequential</td>
<td>Smooth Gradient (Magnitude).</td>
</tr>
<tr>
<td>Quantitative (Relational)</td>
<td>Continuous / Diverging</td>
<td>Two-sided Gradient (Deviation).</td>
</tr>
</tbody>
</table>

## Categorical Data

When processing categorical data, we distinguish between Discrete Scales (the mapping) and Qualitative Palettes (the colors):

- Discrete Scale: This term refers to the function $f: D \rightarrow R$ where the domain $D$ comprises distinct, separate categories, 
such as ```{Sprint, Summer, Autum, Winter}```.
- Qualitative/Categorical Palette: This is the more commonly used term for the range of colors themselves, which refers to a 
set of easily distinguishable colors.
- Categorical Colors: This is the most commonly used informal term for the colors in a Categorical Palette.

### The Challenge of Distinguishability

#### The Magic Number (7 ± 2)

Humans are surprisingly poor at retaining multiple colors in their short-term memory. The rule is simple: once a categorical scale
surpasses 8 to 10 colors, it becomes nearly impossible for a reader to effortlessly switch between the legend and the chart 
without confusion.

#### Perceptual Distance

To make colors "easily distinguishable," they must be far apart in a Perceptually Uniform Color Space (like HCL).

#### The Background Interference (Contrast)

A color’s distinctiveness isn’t absolute; it depends on its surroundings. This phenomenon is called Simultaneous Contrast. 
For instance, a light yellow might be easily noticeable against a dark gray background, but it becomes invisible against a white background.

## Examples of Categorical Palettes

Examples can be seen in [d3-scale-chromatic](https://d3js.org/d3-scale-chromatic)

## Style Guidelines

**Problem:** Select colors to construct an informative and pleasant visual presentation. There is plenty of research work and 
knowledge in this area. One address for style guidelines which is open (free) is the [Data Visualization Style Guidelines of the 
Sunlight Foundation](https://github.com/amycesal/dataviz-style-guide/blob/master/Sunlight-StyleGuide-DataViz.pdf). 

In [ ]:
from color_scales_widget import MyClassicD3Widget
MyClassicD3Widget()